# 💳 신용카드 이상거래 탐지

**목표:** 신용카드 거래 데이터에서 사기 거래를 정확히 탐지한다.

**핵심 문제:** 전체 거래의 0.17%만 사기 → 극심한 불균형 데이터  
→ 단순히 '전부 정상'으로 예측해도 정확도 99.8% → 의미없는 모델  
→ Precision, Recall, F1-Score, AUC-ROC로 평가해야 한다.

**사용 모델:**
- 기초: 로지스틱 회귀, 결정 트리, KNN, 나이브 베이즈, SVM
- 앙상블: 랜덤 포레스트, XGBoost, LightGBM
- 딥러닝: 신경망(MLP)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1단계: 라이브러리 로드
#
# 이 노트북에서 사용하는 외부 패키지들을 한 곳에 모아 불러옵니다.
# 패키지를 미리 import해두면 아래 코드 어디서든 바로 사용할 수 있습니다.
# ─────────────────────────────────────────────────────────────

# 데이터 처리 핵심 라이브러리
import pandas as pd       # 표(DataFrame) 형태의 데이터를 읽고 조작
import numpy as np        # 행렬·배열 연산 (수치 계산 기반)
import matplotlib.pyplot as plt  # 그래프·차트 그리기
import matplotlib         # 한글 폰트 등 전역 설정용
import seaborn as sns     # matplotlib 위에서 동작하는 고급 시각화 라이브러리
import warnings           # 불필요한 경고 메시지 제어용

# 데이터 분리 및 전처리
from sklearn.model_selection import train_test_split  # 훈련/테스트 데이터 분리
from sklearn.preprocessing import StandardScaler      # 피처 정규화 (평균=0, 표준편차=1)

# ── 기초 분류 모델 ──────────────────────────────────────────
from sklearn.linear_model import LogisticRegression   # 로지스틱 회귀: 확률로 0/1 판정
from sklearn.tree import DecisionTreeClassifier        # 결정 트리: 조건 질문을 트리로 쌓아 판정
from sklearn.neighbors import KNeighborsClassifier     # KNN: 가장 비슷한 K개 이웃의 다수결
from sklearn.naive_bayes import GaussianNB             # 나이브 베이즈: 확률 통계 기반 판정
from sklearn.svm import SVC                            # SVM: 클래스를 가장 넓게 나누는 경계선 탐색

# ── 앙상블 모델 (여러 모델을 합쳐 더 강한 예측) ──────────────
from sklearn.ensemble import RandomForestClassifier    # 랜덤 포레스트: 결정 트리 100개의 다수결
import xgboost as xgb    # XGBoost: 이전 트리의 오류를 집중 보정하는 반복 학습
import lightgbm as lgb   # LightGBM: XGBoost와 같은 방식이지만 속도가 더 빠름

# ── 딥러닝 모델 ─────────────────────────────────────────────
from sklearn.neural_network import MLPClassifier  # 신경망(MLP): 뇌 뉴런 구조 모방

# ── 성능 평가 지표 ───────────────────────────────────────────
from sklearn.metrics import (
    confusion_matrix,    # 혼동행렬: 실제 vs 예측 결과를 2×2 표로 정리
    roc_auc_score,       # AUC-ROC: 임계값에 무관한 전체 판별 능력 점수 (1에 가까울수록 좋음)
    roc_curve,           # ROC 커브: 임계값을 바꿀 때 탐지율(TPR) vs 오탐율(FPR) 변화 곡선
    f1_score,            # F1-Score: Precision과 Recall의 조화 평균 (불균형 데이터의 핵심 지표)
    recall_score,        # Recall(탐지율): 실제 사기 중 모델이 잡아낸 비율
    precision_score,     # Precision(정밀도): 사기로 예측한 것 중 실제 사기 비율
)

# ── 불균형 데이터 처리 ───────────────────────────────────────
from imblearn.over_sampling import SMOTE
# SMOTE(Synthetic Minority Over-sampling Technique):
# 소수 클래스(사기 492건)의 데이터를 인공적으로 생성해 다수 클래스(정상)와 균형을 맞춤

# ── 전역 설정 ────────────────────────────────────────────────
# 모든 모델의 random_state를 하나의 상수로 통일
# → 언제 실행해도 동일한 결과가 나오도록 재현성 보장
RANDOM_STATE = 111

matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # 그래프에서 한글이 깨지지 않도록 폰트 설정
matplotlib.rcParams['axes.unicode_minus'] = False      # 마이너스(-) 기호가 □로 나오는 문제 방지
warnings.filterwarnings('ignore')                      # 버전 호환 경고 등 불필요한 메시지 숨김

print('라이브러리 로드 완료!')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2단계: 데이터 로드 및 기본 탐색
#
# CSV 파일을 읽어 데이터의 크기, 컬럼 구성, 클래스 분포를 확인합니다.
# 분석을 시작하기 전에 데이터가 어떻게 생겼는지 파악하는 단계입니다.
# ─────────────────────────────────────────────────────────────

# 컬럼 설명:
#   Time    : 데이터셋 첫 번째 거래로부터 경과한 시간(초). 절대 시각이 아님
#   V1~V28  : 개인정보 보호를 위해 PCA(주성분 분석)로 변환된 익명 특성 28개
#             원본 컬럼명은 공개되지 않음
#   Amount  : 거래 금액 (달러)
#   Class   : 타겟 변수 — 0=정상 거래, 1=사기 거래

df = pd.read_csv('../data/creditcard.csv')  # data/ 폴더에 있는 CSV 파일 읽기

# 데이터 크기 확인: 몇 행(거래 건수) × 몇 열(특성 수)인지 출력
print(f'데이터 크기: {df.shape[0]:,}행 x {df.shape[1]}열')

# 클래스 분포 확인: 정상(0)과 사기(1)가 각각 몇 건인지 확인
# → 이 숫자 차이가 '불균형 데이터' 문제의 핵심
print(f'\n=== 클래스 분포 (불균형 확인) ===')
print(df['Class'].value_counts())

# 전체 거래 중 사기 비율 계산 (mean() = 1의 비율 = 사기 비율)
print(f'\n사기 거래 비율: {df["Class"].mean()*100:.3f}%')

# 처음 5행을 출력해 실제 데이터 형태 확인
df.head()

In [ ]:
# ─────────────────────────────────────────────────────────────
# 클래스 불균형 시각화
#
# 숫자로만 보면 체감이 어려운 불균형 정도를
# 파이 차트와 거래 금액 히스토그램으로 시각화합니다.
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # 그래프 2개를 가로로 나란히 배치

# ── 왼쪽: 파이 차트 (클래스 비율) ───────────────────────────
counts = df['Class'].value_counts()  # 정상/사기 건수 집계
axes[0].pie(
    counts,
    labels=['정상 거래', '사기 거래'],
    autopct='%1.2f%%',              # 각 조각에 소수점 2자리 퍼센트 표시
    colors=['steelblue', 'tomato'], # 정상=파랑, 사기=빨강
    startangle=90                    # 12시 방향부터 시작
)
axes[0].set_title('클래스 분포')

# ── 오른쪽: 거래 금액 히스토그램 (정상 vs 사기 비교) ──────────
# 정상 거래와 사기 거래의 금액 분포가 다른지 확인
# alpha=0.6 으로 겹치는 부분도 보이게 반투명 처리
df[df['Class']==0]['Amount'].hist(ax=axes[1], bins=50, alpha=0.6, color='steelblue', label='정상')
df[df['Class']==1]['Amount'].hist(ax=axes[1], bins=50, alpha=0.6, color='tomato',    label='사기')
axes[1].set_title('거래 금액 분포: 정상 vs 사기')
axes[1].set_xlabel('거래 금액 ($)')
axes[1].legend()
axes[1].set_xlim(0, 500)  # $500 이하 구간만 확대 (대부분의 거래가 이 범위)

plt.tight_layout()  # 두 그래프가 겹치지 않도록 여백 자동 조정
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3단계: 데이터 전처리
#
# 모델이 학습하기 전에 데이터를 '같은 기준'으로 맞추는 작업입니다.
# 예: Amount는 0~25,000 달러 범위지만 V1~V28은 -30~30 수준 → 스케일 차이가 크면
#     모델이 금액이 큰 피처만 중요하게 보는 왜곡이 생깁니다.
#     정규화(StandardScaler)로 모든 피처를 평균=0, 표준편차=1로 맞춥니다.
# ─────────────────────────────────────────────────────────────

# ── 정규화 대상: Amount와 Time ──────────────────────────────
# V1~V28은 이미 PCA(주성분 분석)로 정규화되어 있으므로 건드리지 않음
# Amount와 Time만 따로 StandardScaler로 변환

# 두 컬럼을 하나의 scaler로 순서대로 처리하면 두 번째 fit()이
# 첫 번째 fit() 기준(평균/표준편차)을 덮어써서 Amount 정규화가 망가짐
# → 컬럼마다 독립된 scaler 객체를 사용해야 각 기준이 보존됨
scaler_amount = StandardScaler()  # Amount 전용 정규화 도구
scaler_time   = StandardScaler()  # Time 전용 정규화 도구

# fit_transform: 이 데이터의 평균·표준편차를 계산(fit)하고 바로 변환(transform)
# [['Amount']]처럼 이중 대괄호를 쓰는 이유: scaler는 2차원 배열(행렬)을 기대하기 때문
df['Amount_scaled'] = scaler_amount.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler_time.fit_transform(df[['Time']])

# ── 입력(X)과 정답(y) 분리 ──────────────────────────────────
# 원본 Amount, Time은 이미 정규화된 버전(_scaled)이 추가됐으므로 제외
# Class는 예측해야 할 정답이므로 X에서 제외
features = [col for col in df.columns if col not in ['Class', 'Amount', 'Time']]
X = df[features]  # 입력 피처: V1~V28 + Amount_scaled + Time_scaled (총 30개)
y = df['Class']   # 정답 레이블: 0=정상, 1=사기

# ── 훈련/테스트 데이터 분리 (8:2) ────────────────────────────
# test_size=0.2  : 전체의 20%를 테스트용으로 분리
# random_state   : 동일한 값을 쓰면 항상 같은 방식으로 섞임 → 재현성 보장
# stratify=y     : 분리 후에도 훈련/테스트 세트의 사기 비율(0.17%)이 동일하게 유지됨
#                  stratify 없이 분리하면 우연히 사기가 한쪽에 몰릴 수 있음
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'훈련 데이터: {len(X_train):,}개')
print(f'테스트 데이터: {len(X_test):,}개')
print(f'\n훈련 데이터 내 사기 거래: {y_train.sum()}개 ({y_train.mean()*100:.2f}%)')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4단계: SMOTE로 클래스 불균형 해소
#
# 문제: 훈련 데이터에 정상 227,451건 vs 사기 394건 → 약 578:1 불균형
# 이 상태로 학습하면 모델이 "무조건 정상" 전략을 택해도 손실(loss)이 작아짐
# → 사기를 전혀 잡지 못하는 모델이 만들어짐
#
# 해결책: SMOTE (Synthetic Minority Over-sampling Technique)
#   ① 사기 데이터 1건을 선택
#   ② 그 이웃 K개(기본값=5) 중 하나를 무작위 선택
#   ③ 두 점 사이의 임의 위치에 가상의 사기 데이터를 생성
#   → 단순 복사가 아닌 '새로운 가상 데이터'이므로 과적합 위험이 낮음
#
# ⚠️ 절대 주의: SMOTE는 훈련 데이터에만 적용합니다.
#   테스트 데이터에 적용하면 현실에 없는 가상 데이터로 성능을 평가하게 되어
#   실제 운영 환경과 다른 왜곡된 결과가 나옵니다.
# ─────────────────────────────────────────────────────────────

smote = SMOTE(random_state=RANDOM_STATE)  # random_state 고정으로 재현성 확보

# fit_resample: 훈련 데이터(X_train, y_train)를 입력받아
#              사기 클래스를 증강한 새로운 X_train_bal, y_train_bal을 반환
# _bal 접미사: balanced(균형)의 약자 — SMOTE 적용된 데이터임을 명확히 구분
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('=== SMOTE 적용 전후 비교 ===')
print(f'적용 전 - 정상: {(y_train==0).sum():,}개, 사기: {(y_train==1).sum():,}개')
# 적용 후: 사기를 정상과 동일한 수(227,451건)로 늘림 → 1:1 균형
print(f'적용 후 - 정상: {(y_train_bal==0).sum():,}개, 사기: {(y_train_bal==1).sum():,}개')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5단계: 모델 정의
#
# 9개 분류 모델을 딕셔너리에 담아 관리합니다.
# 딕셔너리 형태(이름: 모델객체)로 저장하면
# 다음 셀의 for 루프에서 코드 중복 없이 모든 모델을 순서대로 학습시킬 수 있습니다.
# ─────────────────────────────────────────────────────────────

models = {
    # ── 기초 모델 ───────────────────────────────────────────

    # 로지스틱 회귀: 각 피처에 가중치를 곱해 합산 → 확률로 변환해 0/1 판정
    # max_iter=1000: 기본값(100)으로는 수렴 안 될 수 있어 넉넉하게 설정
    '로지스틱 회귀': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),

    # 결정 트리: "V14 < -5.0?" 같은 조건 질문을 반복해 트리를 쌓아 분류
    # max_depth=10: 트리가 너무 깊어지면 훈련 데이터만 외우는 과적합 발생 → 깊이 제한
    # class_weight='balanced': 사기 데이터가 적으므로 사기를 잡는 오류에 더 큰 패널티 부여
    '결정 트리': DecisionTreeClassifier(
        random_state=RANDOM_STATE, max_depth=10,
        class_weight='balanced'
    ),

    # KNN (K-최근접 이웃): 새 거래와 가장 비슷한 5개 이웃을 찾아 다수결로 정상/사기 판정
    # n_jobs=-1: 컴퓨터의 모든 CPU 코어를 동시에 사용해 속도 향상
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),

    # 나이브 베이즈: 각 피처가 독립이라고 가정하고 조건부 확률로 클래스를 판정
    # 매우 빠르지만 피처 간 상관관계가 있는 경우 성능이 떨어질 수 있음
    '나이브 베이즈': GaussianNB(),

    # SVM (서포트 벡터 머신): 두 클래스 사이의 여백(margin)을 최대화하는 경계선 탐색
    # kernel='rbf': 선형으로 나눌 수 없는 비선형 데이터도 처리 가능한 커널
    # probability=True: AUC-ROC 계산에 필요한 predict_proba()를 활성화
    'SVM': SVC(
        kernel='rbf',
        class_weight='balanced',
        probability=True,
        random_state=RANDOM_STATE
    ),

    # ── 앙상블 모델 (여러 모델을 결합해 단일 모델보다 강한 예측) ──

    # 랜덤 포레스트: 결정 트리 100개를 각각 다른 데이터/피처 조합으로 학습 → 다수결
    # 단일 결정 트리보다 안정적이고 과적합에 강함
    '랜덤 포레스트': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),

    # XGBoost: 이전 트리가 틀린 데이터에 집중해서 다음 트리를 학습 (부스팅 기법)
    # scale_pos_weight: 불균형 보정 — 정상 수 / 사기 수 비율을 자동으로 계산해 넣음
    #   (y_train==0).sum() / (y_train==1).sum() ≈ 578 → 사기 1건의 가중치를 578배로
    # eval_metric='logloss', verbosity=0: 학습 중 불필요한 로그 출력 억제
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100, random_state=RANDOM_STATE,
        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
        eval_metric='logloss', verbosity=0
    ),

    # LightGBM: XGBoost와 같은 부스팅 방식이지만 리프 단위 성장으로 더 빠름
    # 금융권 실무에서 속도·성능 균형으로 가장 많이 쓰이는 모델
    # verbose=-1: 학습 중 로그 출력 완전히 억제
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=100, random_state=RANDOM_STATE,
        class_weight='balanced', verbose=-1
    ),

    # ── 딥러닝 모델 ────────────────────────────────────────

    # 신경망(MLP, Multi-Layer Perceptron): 뇌 뉴런을 모방한 층(layer) 구조
    # hidden_layer_sizes=(64, 32): 은닉층을 2개 쌓음
    #   입력(30개 피처) → 64개 뉴런 → 32개 뉴런 → 출력(정상/사기)
    # max_iter=50: 최대 50번 학습 반복 (early_stopping이 있으므로 일찍 종료될 수 있음)
    # early_stopping=True: 검증 데이터 성능이 더 이상 개선되지 않으면 조기 종료 → 과적합 방지
    '신경망(MLP)': MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=50, random_state=RANDOM_STATE,
        early_stopping=True
    ),
}

# ── 모델별 훈련 데이터 지정 ──────────────────────────────────
# KNN, 나이브 베이즈, SVM은 SMOTE로 증강된 45만 행짜리 데이터를 쓰면
# 학습 시간이 너무 길거나(KNN/SVM) 메모리 문제가 발생하므로 원본 데이터로 학습
# (이 세 모델은 클래스 불균형 대신 class_weight='balanced' 등으로 보정)
use_original = {'KNN', '나이브 베이즈', 'SVM'}

print('모델 준비 완료! 총', len(models), '개')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6단계: 전체 모델 학습 및 성능 평가
#
# 9개 모델을 for 루프로 순서대로 학습시키고 테스트 데이터로 성능을 측정합니다.
# 측정된 예측값은 pred_cache에 저장해 두어 ROC 커브·혼동행렬 셀에서 재사용합니다.
# (재사용 이유: predict_proba()를 다시 호출하면 시간이 걸리고, 값도 동일하기 때문)
# ─────────────────────────────────────────────────────────────

results    = []      # 각 모델의 성능 지표를 리스트로 수집 → 나중에 DataFrame으로 변환
pred_cache = {}      # {모델명: {y_pred, y_prob, auc}} 형태로 예측값 캐싱

for name, model in models.items():

    # ── 훈련 데이터 선택 ──────────────────────────────────────
    # use_original 집합에 속한 모델(KNN, 나이브 베이즈, SVM)은 원본 사용
    # 나머지(로지스틱 회귀, 결정 트리, 랜덤 포레스트, XGB, LGBM, MLP)는 SMOTE 증강 데이터 사용
    X_tr = X_train if name in use_original else X_train_bal
    y_tr = y_train if name in use_original else y_train_bal

    # ── SVM 전용 샘플링 ───────────────────────────────────────
    # SVM의 학습 시간 복잡도는 O(n²)~O(n³) → 데이터가 2배가 되면 4~8배 느려짐
    # 원본 데이터(약 23만 건)로 학습하면 수십 분이 소요되므로 약 1만 건만 사용
    # 사기:정상 = 1:24 비율로 샘플링해 클래스 불균형을 어느 정도 유지
    if name == 'SVM':
        fraud_idx  = X_tr[y_tr == 1].index                  # 사기 건 전체 인덱스
        normal_idx = X_tr[y_tr == 0].sample(
            n=len(fraud_idx) * 24, random_state=RANDOM_STATE  # 사기의 24배만큼 정상 샘플링
        ).index
        sample_idx = fraud_idx.append(normal_idx)            # 두 인덱스를 합쳐 샘플 구성
        X_tr = X_tr.loc[sample_idx]
        y_tr = y_tr.loc[sample_idx]
        print(f'[SVM] 샘플 크기: {len(X_tr):,}개 (사기 {y_tr.sum()}개 포함)')

    # ── 모델 학습 ─────────────────────────────────────────────
    model.fit(X_tr, y_tr)

    # ── 예측 ──────────────────────────────────────────────────
    # predict(): 임계값 0.5 기준으로 0 또는 1로 확정 예측
    # predict_proba()[:, 1]: '사기일 확률' 값 (0~1 사이 실수)
    #   ROC 커브와 AUC 계산에는 확정 예측이 아닌 확률 값이 필요함
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]  # 열 인덱스 1 = 사기(Class=1) 확률

    # ── 성능 지표 계산 ─────────────────────────────────────────
    # AUC-ROC: 1에 가까울수록 좋음 (임계값에 무관하게 전체적인 판별력 측정)
    auc  = roc_auc_score(y_test, y_prob)

    # F1-Score: Precision과 Recall의 조화 평균
    #   - Precision(정밀도): 사기로 예측한 것 중 실제 사기 비율 → 오탐(FP) 줄이는 지표
    #   - Recall(탐지율): 실제 사기 중 모델이 잡아낸 비율 → 놓침(FN) 줄이는 지표
    #   - 사기 탐지에서는 놓치는 게 더 위험하므로 Recall이 더 중요
    f1   = f1_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)  # 분모=0 예외 방지

    # ── 예측값 캐싱 (ROC 커브·혼동행렬 셀에서 재사용) ──────────
    pred_cache[name] = {'y_pred': y_pred, 'y_prob': y_prob, 'auc': auc}

    # 결과 리스트에 추가 (나중에 DataFrame으로 변환)
    results.append({
        '모델': name,
        'AUC-ROC': round(auc, 4),
        'F1-Score': round(f1, 4),
        'Recall(탐지율)': round(rec, 4),
        'Precision(정밀도)': round(prec, 4),
    })

    print(f'[{name}] AUC={auc:.4f} | F1={f1:.4f} | Recall={rec:.4f} | Precision={prec:.4f}')

print('\n모든 모델 학습 완료!')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 7단계: 성능 비교 표
#
# 앞 셀에서 수집한 results 리스트를 DataFrame으로 변환해 한눈에 비교합니다.
# F1-Score 내림차순 정렬: 불균형 데이터에서 가장 의미 있는 지표를 기준으로 순위를 매깁니다.
# ─────────────────────────────────────────────────────────────

# 리스트 → DataFrame 변환, 모델 이름을 행 인덱스로 설정
results_df = pd.DataFrame(results).set_index('모델')

# F1-Score 기준 내림차순 정렬 (ascending=False = 높은 값이 위로)
results_df = results_df.sort_values('F1-Score', ascending=False)

print('=== 모델 성능 비교 (F1-Score 기준 정렬) ===')
results_df  # Jupyter에서 마지막 줄이 변수/DataFrame이면 표 형태로 자동 렌더링

In [ ]:
# ─────────────────────────────────────────────────────────────
# 8단계: 성능 시각화 — 모델 비교 막대 그래프
#
# F1-Score와 AUC-ROC를 가로 막대 그래프(barh)로 나란히 비교합니다.
# 1위 모델은 빨간색으로 강조해 시각적으로 바로 식별할 수 있게 합니다.
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))  # 그래프 2개를 가로로 나란히 배치

# 1위(인덱스 0) = 빨간색 강조, 나머지 = 파란색
# results_df는 F1-Score 내림차순 정렬된 상태이므로 인덱스 0이 1위 모델
colors = ['tomato' if i == 0 else 'steelblue' for i in range(len(results_df))]

# ── 왼쪽: F1-Score 비교 ──────────────────────────────────────
results_df['F1-Score'].plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title('모델별 F1-Score (사기 탐지 종합 성능)', fontsize=12)
axes[0].set_xlabel('F1-Score')
# 점선(axvline): F1=0.5 기준선 → 이 아래는 동전 던지기 수준의 성능
axes[0].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
# 각 막대 오른쪽에 정확한 수치 표시
for i, v in enumerate(results_df['F1-Score']):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)

# ── 오른쪽: AUC-ROC 비교 ─────────────────────────────────────
results_df['AUC-ROC'].plot(kind='barh', ax=axes[1], color=colors)
axes[1].set_title('모델별 AUC-ROC (전체 판별 능력)', fontsize=12)
axes[1].set_xlabel('AUC-ROC')
# xlim을 0.5~1.0으로 설정: AUC는 0.5가 랜덤 분류기 기준이므로 차이를 더 잘 보이게 확대
axes[1].set_xlim(0.5, 1.0)
for i, v in enumerate(results_df['AUC-ROC']):
    axes[1].text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()  # 두 그래프가 겹치지 않도록 여백 자동 조정
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# 9단계: ROC 커브 비교 (전체 모델)
#
# ROC 커브란?
#   - 임계값(threshold)을 0→1로 변화시키면서
#     X축: FPR (False Positive Rate, 정상을 사기로 잘못 탐지하는 비율)
#     Y축: TPR (True Positive Rate, 실제 사기를 올바르게 탐지하는 비율)
#     을 플롯한 곡선입니다.
#   - 왼쪽 상단 모서리에 가까울수록 좋은 모델 (FPR 낮고 TPR 높음)
#   - 대각선(점선)은 랜덤 분류기(AUC=0.5) 기준선
#
# AUC(Area Under Curve): ROC 커브 아래 면적 → 1에 가까울수록 좋음
# ─────────────────────────────────────────────────────────────

plt.figure(figsize=(10, 7))

# 모델마다 고유 색상 지정 (여러 선이 겹쳐도 구분 가능하도록)
color_map = {
    '로지스틱 회귀': 'royalblue', '결정 트리': 'orange',
    'KNN': 'green', '나이브 베이즈': 'purple', 'SVM': 'teal',
    '랜덤 포레스트': 'red', 'XGBoost': 'brown',
    'LightGBM': 'deeppink', '신경망(MLP)': 'gray',
}

for name in models:
    # cell-6(학습 셀)에서 pred_cache에 저장해둔 확률값과 AUC 재사용
    # → 이 셀을 단독 실행하면 KeyError 발생 (학습 셀을 먼저 실행해야 함)
    y_prob = pred_cache[name]['y_prob']
    auc    = pred_cache[name]['auc']

    # roc_curve(): 확률값(y_prob)을 기반으로 다양한 임계값에서 fpr, tpr 계산
    # _는 임계값 배열인데 여기서는 사용하지 않으므로 무시
    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # 범례에 모델명과 AUC 값을 함께 표시해 한눈에 비교 가능하도록
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color_map[name])

# 대각선 점선: 랜덤으로 찍는 분류기의 ROC 커브 (이 선 아래면 랜덤보다 나쁜 모델)
plt.plot([0, 1], [0, 1], 'k--', label='랜덤 분류기 (AUC=0.5)')

plt.title('전체 모델 ROC 커브 비교', fontsize=13)
plt.xlabel('False Positive Rate (정상을 사기로 오탐하는 비율)')
plt.ylabel('True Positive Rate (실제 사기를 탐지하는 비율)')
plt.legend(loc='lower right', fontsize=9)  # 오른쪽 하단에 범례 배치 (커브와 겹치지 않음)
plt.grid(True, alpha=0.3)   # 격자선으로 값 읽기 쉽게, alpha=0.3으로 흐리게
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# 10단계: 최고 성능 모델 상세 분석
#
# F1-Score 1위 모델에 대해 두 가지 추가 분석을 수행합니다.
#   1) 혼동행렬(Confusion Matrix): 예측 결과를 4가지 경우로 정리
#   2) 특성 중요도(Feature Importance): 어떤 피처가 판단에 영향을 많이 줬는지
# ─────────────────────────────────────────────────────────────

# results_df는 F1-Score 내림차순 정렬 → 인덱스 0이 1위 모델
best_model_name = results_df.index[0]
best_model      = models[best_model_name]  # 학습 완료된 모델 객체
print(f'최고 성능 모델: {best_model_name}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 왼쪽: 혼동행렬 ──────────────────────────────────────────
# 혼동행렬의 4칸 의미:
#   TN (True Negative)  : 실제 정상 → 정상으로 예측 ✅ (정상 거래를 올바르게 통과)
#   FP (False Positive) : 실제 정상 → 사기로 예측  ❌ (정상 거래를 사기로 오탐 → 고객 불편)
#   FN (False Negative) : 실제 사기 → 정상으로 예측 ❌ (사기를 놓침 → 금전 손실, 더 위험!)
#   TP (True Positive)  : 실제 사기 → 사기로 예측  ✅ (사기를 올바르게 탐지)
#
# 사기 탐지에서는 FN(놓침)이 FP(오탐)보다 훨씬 치명적 → Recall을 높이는 게 중요

# pred_cache에서 1위 모델의 예측값 재사용 (학습 셀에서 저장해둠)
y_pred_best = pred_cache[best_model_name]['y_pred']
cm = confusion_matrix(y_test, y_pred_best)

# annot=True: 각 칸에 숫자 표시, fmt='d': 정수 형식
# cmap='Blues': 파란 색조 — 값이 클수록 진한 파란색
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['예측:정상', '예측:사기'],
            yticklabels=['실제:정상', '실제:사기'])
axes[0].set_title(f'혼동행렬 - {best_model_name}')

# ── 오른쪽: 특성 중요도 ──────────────────────────────────────
# feature_importances_: 트리 기반 모델(결정 트리, 랜덤 포레스트, XGBoost, LightGBM)만 지원
# KNN, 나이브 베이즈, 로지스틱 회귀, SVM, MLP는 이 속성이 없음
if hasattr(best_model, 'feature_importances_'):
    # 피처명과 중요도를 DataFrame으로 묶어 내림차순 정렬 → 상위 15개만 표시
    importance_df = pd.DataFrame({
        '특성': features,
        '중요도': best_model.feature_importances_
    }).sort_values('중요도', ascending=False).head(15)

    # palette='viridis': 중요도 순서에 따라 색상 그라데이션
    sns.barplot(data=importance_df, x='중요도', y='특성', palette='viridis', ax=axes[1])
    axes[1].set_title(f'특성 중요도 (상위 15개) - {best_model_name}')
else:
    # 특성 중요도를 지원하지 않는 모델일 경우 안내 텍스트 표시
    axes[1].text(0.5, 0.5, f'{best_model_name}은\n특성 중요도를 제공하지 않습니다',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=12)
    axes[1].axis('off')  # 축 눈금 등 불필요한 요소 숨김

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# 11단계: 최종 결론
#
# 전체 파이프라인의 결과를 요약하고 핵심 인사이트를 정리합니다.
# ─────────────────────────────────────────────────────────────

print('=' * 55)
print('           최종 모델 성능 요약')
print('=' * 55)
# to_string(): DataFrame을 텍스트 형태로 출력 (Jupyter 표 렌더링 대신 print에 맞게)
print(results_df.to_string())
print('=' * 55)

# F1-Score 1위 모델의 이름과 지표 출력
print(f'\n최고 성능 모델: {results_df.index[0]}')
print(f'  F1-Score : {results_df["F1-Score"].iloc[0]:.4f}')   # iloc[0] = 첫 번째 행(1위)
print(f'  AUC-ROC  : {results_df["AUC-ROC"].iloc[0]:.4f}')
print()

# ── 프로젝트 핵심 인사이트 ─────────────────────────────────
print('[인사이트]')
# 앙상블 vs 기초: 하이퍼파라미터 튜닝 없이도 앙상블이 일반적으로 우세
# 단, 이번 결과에서는 KNN이 1위 → 튜닝 없는 앙상블도 한계 있음을 보여줌
print('  - 앙상블 모델(RF, XGB, LGBM)이 기초 모델 대비 F1 높음')
# 나이브 베이즈: 속도는 빠르지만 피처 독립 가정이 이 데이터엔 맞지 않아 F1 낮음
print('  - 나이브 베이즈: 빠르지만 불균형에 취약')
# KNN: 유사 이웃을 찾기 위해 모든 데이터를 참조 → 데이터가 클수록 느려짐
print('  - KNN: 대용량 데이터에서 속도 문제')
# 핵심 메시지: 불균형 데이터에서 Accuracy는 의미 없음
# "무조건 정상"이라 해도 99.8% 정확도 → 사기 탐지 실패 모델
# 반드시 F1-Score(정밀도+탐지율 균형)와 AUC-ROC(전체 판별력)로 평가해야 함
print('  - Accuracy가 아닌 F1/AUC로 평가해야 불균형 데이터에서 의미 있음')